# 9.1 Событийная модель: события, сессии, пользователи — код страницы

Аналитика 101 · блок «Дата аналитика» · тема 09 «Продуктовые метрики и события» · страница 9.1.

В ноутбуке два блока кода страницы на pandas: сессии через `diff` и `cumsum`, затем склейка устройств и витрина «пользователь — день». Третий блок страницы — тот же расчёт сессий на SQL — лежит рядом файлом `sessions.sql`, ему нужен PostgreSQL. В конце — лента героя из интерактива: 21 строка и все 28 состояний ползунка и переключателя. Данные учебные: журнал событий из общего набора темы `data/events/`.

**Как работать.** Выполняйте ячейки по порядку сверху вниз: второй блок продолжает первый.

Вывод получен на Python 3.13, pandas 3.0 и pyarrow 26.

## Где лежат данные

Код страницы читает журнал по пути от корня папки с материалами: `data/events/events.parquet`. Ноутбук лежит в папке страницы, поэтому первая ячейка поднимается до папки, в которой есть `data/`, и делает её рабочей. Папку самого ноутбука запоминаем: рядом с ним лежит `hero_12_15_may.csv`.

In [1]:
import os
import pathlib
import platform

import pandas as pd
import pyarrow

here = pathlib.Path.cwd()                   # папка ноутбука: здесь лежит hero_12_15_may.csv
root = here
while not (root / "data" / "events").is_dir() and root != root.parent:
    root = root.parent                      # поднимаемся, пока рядом не окажется data/events/
assert (root / "data" / "events" / "events.parquet").exists(), \
    "Не найдена папка data/events/. Скачайте общий набор темы 09 и положите data/ рядом с папкой страницы"
os.chdir(root)

print("Python", platform.python_version(), "| pandas", pd.__version__, "| pyarrow", pyarrow.__version__)

Python 3.13.18 | pandas 3.0.5 | pyarrow 26.0.1


## Код: сессии и витрина

### То же в pandas: diff внутри groupby и cumsum

In [2]:
import pandas as pd

ev = pd.read_parquet("data/events/events.parquet")  # сырой журнал, время в UTC

# 1. Очищенный слой: без дублей event_id, без ботов, день по Минску
ev = ev.drop_duplicates("event_id")
ev["day"] = ev["event_time"].dt.tz_convert("Europe/Minsk").dt.date
client = ev[ev["origin"] == "client"]
per_day = client.groupby(["anonymous_id", "day"]).size()
bots = per_day[per_day > 100].index.get_level_values("anonymous_id").unique()
ev = ev[~ev["anonymous_id"].isin(bots)]
print("строк:", len(ev), "| ботов:", len(bots))

# 2. Сессии: пауза длиннее порога на том же устройстве начинает новую
def sessionize(ev, minutes):
    c = (ev[ev["origin"] == "client"]
         .sort_values(["anonymous_id", "event_time", "event_id"]))
    gap = c.groupby("anonymous_id")["event_time"].diff()          # как LAG
    is_new = gap.isna() | (gap > pd.Timedelta(minutes=minutes))
    session_no = is_new.groupby(c["anonymous_id"]).cumsum()      # как SUM() OVER
    return c.assign(gap=gap, is_new=is_new, session_no=session_no)

for m in (30, 5):
    s = sessionize(ev, m)
    print(f"порог {m:>2} мин: устройств {s['anonymous_id'].nunique()}, "
          f"сессий {s['is_new'].sum()}")

hero = sessionize(ev, 30).query("anonymous_id in ['d_f6f77955', 'd_398d33df']")
hero = hero[hero["event_time"] >= "2026-05-12 00:00+03:00"]
minsk = hero["event_time"].dt.tz_convert("Europe/Minsk").dt.strftime("%d.%m %H:%M")
print(hero.assign(minsk=minsk)
          .groupby(["anonymous_id", "session_no"])
          .agg(start=("minsk", "first"), events=("event_id", "size")))

строк: 303162 | ботов: 6


порог 30 мин: устройств 10619, сессий 57267


порог  5 мин: устройств 10619, сессий 66895


                               start  events
anonymous_id session_no                     
d_398d33df   1           13.05 12:31       3
             2           15.05 20:10       4
d_f6f77955   3           12.05 08:05       8
             4           12.05 19:40       4


Счётчики совпадают с SQL: 10 619 устройств, 57 267 сессий при 30 минутах и 66 895 при 5.

### Склейка и витрина «пользователь — день»

In [3]:
# продолжение: ev — очищенный слой, sessionize — из блока выше
# 3. Склейка: каждому устройству — аккаунт, который на нём входил
dev_user = (ev.dropna(subset=["anonymous_id", "user_id"])
              .groupby("anonymous_id")["user_id"].first())
ev["person"] = (ev["user_id"]
                .fillna(ev["anonymous_id"].map(dev_user))
                .fillna(ev["anonymous_id"]))           # входа не было: человек = устройство
print("людей:", ev["person"].nunique())

# 4. Витрина «пользователь — день»: одна строка на человека и минский день
s = sessionize(ev, 30)
paid = ev[ev["event_name"] == "order_paid"]                # заказы — только с сервера
user_day = (
    ev.groupby(["person", "day"]).size().rename("events").to_frame()
      .join(s[s["is_new"]].groupby(["person", "day"]).size().rename("sessions"))
      .join(paid.groupby(["person", "day"])
                .agg(orders=("order_id", "size"), revenue=("revenue", "sum")))
      .fillna(0).astype(int)
      .reset_index()
)
days = user_day["day"].astype(str)
user_day = user_day[(days >= "2026-01-05") & (days <= "2026-06-28")]  # полные дни
print(user_day.shape)
print(user_day.head(3))
print(user_day[user_day["person"] == "u_06883"])

# DAU — просто число строк витрины за день
dau = user_day.groupby("day").size()
print("DAU в мае:", round(dau[dau.index.astype(str).str[:7] == "2026-05"].mean(), 1))

людей: 10413


(45250, 6)
       person         day  events  sessions  orders  revenue
0  d_00044e27  2026-04-27       5         1       0        0
1  d_00044e27  2026-05-07       8         2       0        0
2  d_00044e27  2026-05-08       6         1       0        0
        person         day  events  sessions  orders  revenue
45252  u_06883  2026-04-20       4         1       0        0
45253  u_06883  2026-05-03       2         1       0        0
45254  u_06883  2026-05-12      13         2       1     4600
45255  u_06883  2026-05-13       3         1       0        0
45256  u_06883  2026-05-15       5         1       1     1900
DAU в мае: 339.3


Сессия засчитывается дню, в который началась. В первых строках витрины — анонимное устройство: аккаунта у него нет, и человек = устройство.

## Лента героя: 28 состояний интерактива

В интерактиве страницы — 21 строка героя за 12–15 мая, ползунок порога неактивности и переключатель склейки. Ниже тот же расчёт для всех 14 значений порога, без склейки и со склейкой. Правило страницы: сессия считается по устройству; серверный `order_paid` относим к последней сессии того же человека, начатой до оплаты; без склейки человек — `user_id` строки, а если его нет — `anonymous_id`.

In [4]:
hero_rows = pd.read_csv(here / "hero_12_15_may.csv", parse_dates=["event_time"])
print(len(hero_rows), "строк:", hero_rows["origin"].value_counts().to_dict())


def hero_state(h, minutes, stitch):
    """Сессии, сессии с заказом, люди и покупатели на ленте героя."""
    account = h.dropna(subset=["anonymous_id", "user_id"]).groupby("anonymous_id")["user_id"].first()
    who = h["user_id"].fillna(h["anonymous_id"].map(account) if stitch else h["anonymous_id"]).fillna(h["anonymous_id"])
    h = h.assign(who=who)
    c = h[h["origin"] == "client"].sort_values(["anonymous_id", "event_time", "event_id"])
    gap = c.groupby("anonymous_id")["event_time"].diff()
    c = c.assign(session_no=(gap.isna() | (gap > pd.Timedelta(minutes=minutes))).groupby(c["anonymous_id"]).cumsum())
    sess = c.groupby(["anonymous_id", "session_no"]).agg(start=("event_time", "min"), people=("who", set))
    paid = h[h["event_name"] == "order_paid"]
    with_order = {sess[(sess["start"] <= o.event_time) & sess["people"].map(lambda p: o.who in p)]["start"].idxmax()
                  for o in paid.itertuples()}
    return len(sess), len(with_order), h["who"].nunique(), paid["who"].nunique()


print("порог, мин | без склейки: сессий, с заказом, людей, покупателей | со склейкой: то же")
for minutes in [1, 2, 5, 10, 15, 20, 30, 45, 60, 120, 240, 480, 720, 1440]:
    a, b = hero_state(hero_rows, minutes, False), hero_state(hero_rows, minutes, True)
    print(f"{minutes:>10} | {a[0]:>2} {a[1]} {a[2]} {a[3]} | {b[0]:>2} {b[1]} {b[2]} {b[3]}")

21 строк: {'client': 19, 'server': 2}
порог, мин | без склейки: сессий, с заказом, людей, покупателей | со склейкой: то же
         1 | 10 2 2 1 | 10 2 1 1
         2 |  5 2 2 1 |  5 2 1 1
         5 |  5 2 2 1 |  5 2 1 1


        10 |  5 2 2 1 |  5 2 1 1


        15 |  4 2 2 1 |  4 2 1 1
        20 |  4 2 2 1 |  4 2 1 1
        30 |  4 2 2 1 |  4 2 1 1
        45 |  4 2 2 1 |  4 2 1 1
        60 |  4 2 2 1 |  4 2 1 1


       120 |  4 2 2 1 |  4 2 1 1


       240 |  4 2 2 1 |  4 2 1 1
       480 |  4 2 2 1 |  4 2 1 1
       720 |  3 2 2 1 |  3 2 1 1
      1440 |  3 2 2 1 |  3 2 1 1


Порог меняет только число сессий: от 10 при одной минуте до 3 при 12 часах. Склейка меняет только число людей: без неё их двое, и конверсия людей в покупку — 50% вместо 100%.